#Init

#Business Transformation & Modeling

In [0]:
%sql
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_number,
    ci.first_name,
    ci.last_name
FROM workspace.silver.crm_customers ci

#Write it to Gold Table

#The Transformation Logic


In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY pn.start_date, pn.product_number) AS product_key, -- Surrogate key
    pn.product_id,
    pn.product_number,
    pn.product_name,
    pn.category_id,
    pc.category,
    pc.subcategory,
    pc.maintenance_flag,
    pn.product_line,
    pn.start_date
FROM silver.crm_products pn
LEFT JOIN silver.erp_product_category pc
    ON pn.category_id = pc.category_id
--WHERE pn.end_date IS NULL; -- Filter out all historical data
"""
df = spark.sql(query)


In [0]:
df.display()

In [0]:
df.limit(10).display()

#Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_products")

In [0]:
df.display()

#The Transformation Logic

In [0]:
query = """
SELECT
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM silver.crm_sales sd
LEFT JOIN gold.dim_products pr
    ON sd.product_number = pr.product_number
LEFT JOIN gold.dim_customers cu
    ON sd.customer_id = cu.customer_id;
"""
df = spark.sql(query)


In [0]:
df.limit(10).display()

#Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.fact_sales")
     

In [0]:
df.display()